## import packages

In [1]:
import torch
import numpy as np
import random
import torch.nn.functional as F

## reproducible

In [2]:
# reproducible
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(888)

## gate

In [3]:
seq_length = 5
num_experts = 8

In [4]:
router_logits = torch.randn(seq_length, num_experts)
router_logits

tensor([[-0.0767, -1.1224, -0.1318, -1.8342,  0.5491, -1.6664, -0.9915,  0.0562],
        [-0.3024, -1.0513, -0.5106,  0.1460, -1.6237,  0.5749,  1.4396,  1.0438],
        [-0.0440,  0.1769, -1.2321,  1.5142,  0.1770,  0.7858,  1.2745, -1.0547],
        [ 1.3667, -1.2219, -0.9361,  0.6755,  0.6861,  0.4226, -0.8953, -0.6651],
        [ 0.1580,  0.0663,  0.1625, -1.3114,  2.2724, -1.0972,  0.8258,  0.3806]])

In [5]:
routing_weights = F.softmax(router_logits, dim=1, dtype=torch.float)
routing_weights

tensor([[0.1643, 0.0577, 0.1555, 0.0283, 0.3072, 0.0335, 0.0658, 0.1876],
        [0.0622, 0.0294, 0.0505, 0.0974, 0.0166, 0.1496, 0.3552, 0.2391],
        [0.0669, 0.0835, 0.0204, 0.3179, 0.0835, 0.1534, 0.2501, 0.0244],
        [0.3563, 0.0268, 0.0356, 0.1785, 0.1804, 0.1386, 0.0371, 0.0467],
        [0.0670, 0.0612, 0.0673, 0.0154, 0.5554, 0.0191, 0.1307, 0.0838]])

## select top_k experts

In [6]:
top_k = 2
routing_weights, selected_experts = torch.topk(routing_weights, k=top_k, dim=-1)
print(routing_weights)
print(selected_experts)

tensor([[0.3072, 0.1876],
        [0.3552, 0.2391],
        [0.3179, 0.2501],
        [0.3563, 0.1804],
        [0.5554, 0.1307]])
tensor([[4, 7],
        [6, 7],
        [3, 6],
        [0, 4],
        [4, 6]])


In [7]:
routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
routing_weights

tensor([[0.6208, 0.3792],
        [0.5977, 0.4023],
        [0.5596, 0.4404],
        [0.6639, 0.3361],
        [0.8095, 0.1905]])

## get expert mask

In [8]:
expert_mask = torch.nn.functional.one_hot(selected_experts, num_classes=num_experts)
# x: seq_len, y: top_k, z: num_experts
expert_mask

tensor([[[0, 0, 0, 0, 1, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 1]],

        [[0, 0, 0, 0, 0, 0, 1, 0],
         [0, 0, 0, 0, 0, 0, 0, 1]],

        [[0, 0, 0, 1, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 1, 0]],

        [[1, 0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 1, 0, 0, 0]],

        [[0, 0, 0, 0, 1, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 1, 0]]])

In [9]:
expert_mask.shape

torch.Size([5, 2, 8])

In [10]:
expert_mask = expert_mask.permute(2, 1, 0)
# x: num_experts, y: top_k, z: seq_len
expert_mask

tensor([[[0, 0, 0, 1, 0],
         [0, 0, 0, 0, 0]],

        [[0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0]],

        [[0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0]],

        [[0, 0, 1, 0, 0],
         [0, 0, 0, 0, 0]],

        [[1, 0, 0, 0, 1],
         [0, 0, 0, 1, 0]],

        [[0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0]],

        [[0, 1, 0, 0, 0],
         [0, 0, 1, 0, 1]],

        [[0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0]]])

In [11]:
expert_mask.shape

torch.Size([8, 2, 5])

## find activated experts

In [12]:
expert_hitted = torch.greater(expert_mask.sum(dim=(-1, -2)), 0).nonzero()
expert_hitted

tensor([[0],
        [3],
        [4],
        [6],
        [7]])

## compute the expert hidden state

In [13]:
for expert_idx in expert_hitted:
    print("----------------------------------")
    print(expert_mask[expert_idx])
    print(expert_mask[expert_idx].squeeze(0))

----------------------------------
tensor([[[0, 0, 0, 1, 0],
         [0, 0, 0, 0, 0]]])
tensor([[0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0]])
----------------------------------
tensor([[[0, 0, 1, 0, 0],
         [0, 0, 0, 0, 0]]])
tensor([[0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0]])
----------------------------------
tensor([[[1, 0, 0, 0, 1],
         [0, 0, 0, 1, 0]]])
tensor([[1, 0, 0, 0, 1],
        [0, 0, 0, 1, 0]])
----------------------------------
tensor([[[0, 1, 0, 0, 0],
         [0, 0, 1, 0, 1]]])
tensor([[0, 1, 0, 0, 0],
        [0, 0, 1, 0, 1]])
----------------------------------
tensor([[[0, 0, 0, 0, 0],
         [1, 1, 0, 0, 0]]])
tensor([[0, 0, 0, 0, 0],
        [1, 1, 0, 0, 0]])


In [14]:
# torch.where return row and column index of non-zero elements
# so, idx means top_k expert, top_x means ith token
for expert_idx in expert_hitted:
    idx, top_x = torch.where(expert_mask[expert_idx].squeeze(0))
    print(expert_idx, idx, top_x)
# for example, tensor([0]) tensor([0]) tensor([3]) means:
# expert0 is the top0 expert for token3

tensor([0]) tensor([0]) tensor([3])
tensor([3]) tensor([0]) tensor([2])
tensor([4]) tensor([0, 0, 1]) tensor([0, 4, 3])
tensor([6]) tensor([0, 1, 1]) tensor([1, 2, 4])
tensor([7]) tensor([1, 1]) tensor([0, 1])


In [15]:
hidden_dim = 8
hidden_states = torch.randn(seq_length, hidden_dim)
hidden_states

tensor([[ 0.3185, -0.8280, -0.4221,  1.2083, -0.9162, -0.4851,  0.1958, -1.2579],
        [ 1.2721, -0.1076,  0.5810, -0.5034,  1.2006,  1.0950, -0.6355, -0.3710],
        [-0.7305, -0.2040, -0.4935,  1.0729,  0.5931,  0.9913, -0.8443, -1.7649],
        [-0.7167, -1.3267, -1.3304,  0.3669, -0.0958, -0.7035, -1.0582,  0.0507],
        [-0.3311, -0.9611, -0.4659,  1.1254, -0.2185, -1.0000,  0.8923, -1.8942]])

tensor([4]) tensor([0, 0, 1]) tensor([0, 4, 3])

In [16]:
expert_idx = 4
idx = [0, 0, 1]
top_x = [0, 4, 3]

### index the correct hidden states for expert4

In [17]:
current_state = hidden_states[None, top_x]
current_state

tensor([[[ 0.3185, -0.8280, -0.4221,  1.2083, -0.9162, -0.4851,  0.1958,
          -1.2579],
         [-0.3311, -0.9611, -0.4659,  1.1254, -0.2185, -1.0000,  0.8923,
          -1.8942],
         [-0.7167, -1.3267, -1.3304,  0.3669, -0.0958, -0.7035, -1.0582,
           0.0507]]])

In [18]:
current_state = current_state.reshape(-1, hidden_dim)
current_state

tensor([[ 0.3185, -0.8280, -0.4221,  1.2083, -0.9162, -0.4851,  0.1958, -1.2579],
        [-0.3311, -0.9611, -0.4659,  1.1254, -0.2185, -1.0000,  0.8923, -1.8942],
        [-0.7167, -1.3267, -1.3304,  0.3669, -0.0958, -0.7035, -1.0582,  0.0507]])

### routing_weights

In [19]:
routing_weights

tensor([[0.6208, 0.3792],
        [0.5977, 0.4023],
        [0.5596, 0.4404],
        [0.6639, 0.3361],
        [0.8095, 0.1905]])

In [20]:
routing_weights[top_x, idx, None]

tensor([[0.6208],
        [0.8095],
        [0.3361]])

### compute

In [21]:
def expert4_layer(x: torch.Tensor) -> torch.Tensor:
    return x

In [22]:
current_hidden_states = expert4_layer(current_state) * routing_weights[top_x, idx, None]
current_hidden_states

tensor([[ 0.1977, -0.5140, -0.2621,  0.7501, -0.5688, -0.3011,  0.1216, -0.7809],
        [-0.2680, -0.7780, -0.3772,  0.9110, -0.1769, -0.8095,  0.7223, -1.5333],
        [-0.2409, -0.4460, -0.4472,  0.1233, -0.0322, -0.2365, -0.3557,  0.0170]])

In [23]:
print(0.6208 * current_state[0])
print(0.8095 * current_state[1])
print(0.3361 * current_state[2])

tensor([ 0.1977, -0.5140, -0.2621,  0.7501, -0.5688, -0.3011,  0.1216, -0.7809])
tensor([-0.2680, -0.7780, -0.3772,  0.9110, -0.1769, -0.8095,  0.7223, -1.5334])
tensor([-0.2409, -0.4459, -0.4471,  0.1233, -0.0322, -0.2364, -0.3557,  0.0170])


## combine

In [24]:
final_hidden_states = torch.zeros(
    hidden_states.shape,
    dtype=hidden_states.dtype,
    device=hidden_states.device,
)

In [27]:
final_hidden_states.index_add_(0, torch.tensor(top_x), current_hidden_states.to(hidden_states.dtype))
final_hidden_states

tensor([[ 0.3954, -1.0281, -0.5241,  1.5002, -1.1376, -0.6023,  0.2431, -1.5618],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-0.4818, -0.8919, -0.8944,  0.2467, -0.0644, -0.4729, -0.7114,  0.0341],
        [-0.5361, -1.5559, -0.7543,  1.8219, -0.3537, -1.6190,  1.4446, -3.0667]])